# Python Mastery — Module 4
## Statistics and Probability for Data Science

**Goal:** Build the statistical foundation needed to understand data, evaluate experiments, reason about uncertainty, and prepare for machine learning.

### Learning outcomes
By the end, you should be able to:
- Summarize data with descriptive statistics and visualizations.
- Understand probability, conditional probability, and Bayes' theorem.
- Explain sampling, sampling bias, and the Central Limit Theorem.
- Interpret confidence intervals, hypothesis tests, p-values, and effect sizes.
- Distinguish correlation from causation.
- Understand covariance and the foundations of linear regression.
- Recognize common statistical mistakes in business and data-science work.

**How to study:** Read the explanation, run the example, then answer the practice question before looking at the suggested solution. Examples use synthetic data.

## 1. Population, sample, parameter, and statistic

- **Population:** the entire group we want to understand.
- **Sample:** the subset we actually observe.
- **Parameter:** a numerical property of the population, such as its true mean.
- **Statistic:** a number calculated from a sample, such as the sample mean.

We usually sample because measuring the entire population is expensive, slow, or impossible. The goal is not simply to calculate a number; it is to understand how uncertain that number is.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
population = rng.normal(loc=500, scale=120, size=100_000)

sample = rng.choice(population, size=500, replace=False)

print(f"Population mean: {population.mean():.2f}")
print(f"Sample mean:     {sample.mean():.2f}")
print(f"Population SD:   {population.std(ddof=0):.2f}")
print(f"Sample SD:       {sample.std(ddof=1):.2f}")

## 2. Descriptive statistics

Descriptive statistics summarize the data you already have.

- **Mean:** arithmetic average; sensitive to extreme values.
- **Median:** middle value; often more robust when data are skewed.
- **Mode:** most frequent value or category.
- **Range:** maximum minus minimum.
- **Variance:** average squared deviation from the mean.
- **Standard deviation (SD):** square root of variance, in the original unit.
- **Quartiles and IQR:** Q1 is the 25th percentile, Q3 the 75th percentile, and IQR = Q3 − Q1.

For a sample, the common sample variance divides the sum of squared deviations by `n - 1` (`ddof=1`). This corrects bias in estimating population variance from a sample.

In [ ]:
costs = pd.Series([80, 95, 100, 105, 120, 130, 150, 900], name="claim_cost")
summary = {
    "mean": costs.mean(),
    "median": costs.median(),
    "sample_variance": costs.var(ddof=1),
    "sample_sd": costs.std(ddof=1),
    "Q1": costs.quantile(0.25),
    "Q3": costs.quantile(0.75),
    "IQR": costs.quantile(0.75) - costs.quantile(0.25),
}
pd.Series(summary).round(2)

In [ ]:
plt.figure()
plt.hist(costs, bins=8)
plt.title("Distribution of synthetic claim costs")
plt.xlabel("Claim cost")
plt.ylabel("Frequency")
plt.show()

**Interpretation:** The high value of 900 pulls the mean upward much more than it moves the median. In skewed cost data, it is often useful to report both mean and median, rather than assuming one tells the whole story.

## 3. Probability essentials

Probability is a number between 0 and 1:
- 0 means impossible under the model.
- 1 means certain under the model.
- The complement rule: `P(not A) = 1 - P(A)`.
- For mutually exclusive events: `P(A or B) = P(A) + P(B)`.
- In general: `P(A or B) = P(A) + P(B) - P(A and B)`.

Events are **independent** when knowing one occurred does not change the probability of the other: `P(A and B) = P(A)P(B)`.

In [ ]:
# A fair six-sided die
p_even = 3 / 6
p_not_even = 1 - p_even
p_six = 1 / 6
p_even_or_six = p_even + p_six - (1 / 6)  # Six is already an even outcome.

print("P(even):", p_even)
print("P(not even):", p_not_even)
print("P(even or six):", p_even_or_six)

## 4. Conditional probability and Bayes' theorem

Conditional probability asks: what is the probability of A **given that** B has occurred?

\[
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
\]

Bayes' theorem reverses the condition:

\[
P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}
\]

This matters in fraud detection, screening tests, risk models, and alerts. Even a test with good sensitivity can produce many false positives when the event being detected is rare.

In [ ]:
# Example: a screening test for a rare condition.
prevalence = 0.01
sensitivity = 0.90        # P(positive | condition)
specificity = 0.95        # P(negative | no condition)
false_positive_rate = 1 - specificity

p_positive = sensitivity * prevalence + false_positive_rate * (1 - prevalence)
p_condition_given_positive = sensitivity * prevalence / p_positive

print(f"Probability of a positive result: {p_positive:.3%}")
print(f"Probability of condition given positive result: {p_condition_given_positive:.3%}")

**Key lesson:** The probability of a condition after a positive result is not the same as test sensitivity. Always account for prevalence and false positives. This is an illustrative calculation, not medical advice.

## 5. Common probability distributions

### Bernoulli
One trial with two outcomes, often coded 0/1. Example: whether a claim is fraudulent.

### Binomial
Number of successes in a fixed number of independent Bernoulli trials with the same success probability.

### Normal
Symmetric, bell-shaped distribution described by a mean and standard deviation. Many measurement errors are approximately normal, but not all real-world variables are.

### Poisson
Counts of events in a fixed interval, under assumptions including a stable rate and independence. It is not a universal model for all counts.

### Uniform
All values in a defined interval have equal density (continuous case).

Choose a distribution based on the data-generating process and check whether its assumptions are reasonable.

In [ ]:
rng = np.random.default_rng(7)
bernoulli = rng.binomial(n=1, p=0.15, size=1000)
binomial = rng.binomial(n=10, p=0.15, size=1000)
normal = rng.normal(loc=0, scale=1, size=1000)
poisson = rng.poisson(lam=3, size=1000)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].hist(bernoulli, bins=[-0.5, 0.5, 1.5], rwidth=0.8)
axes[0, 0].set_title("Bernoulli outcomes")
axes[0, 1].hist(binomial, bins=np.arange(-0.5, 11.5, 1), rwidth=0.8)
axes[0, 1].set_title("Binomial successes out of 10")
axes[1, 0].hist(normal, bins=25)
axes[1, 0].set_title("Normal values")
axes[1, 1].hist(poisson, bins=np.arange(-0.5, max(poisson) + 1.5, 1), rwidth=0.8)
axes[1, 1].set_title("Poisson counts")
plt.tight_layout()
plt.show()

## 6. Sampling, bias, and the Central Limit Theorem

A sample is useful only if it reasonably represents the population relevant to the question.

Common problems:
- **Selection bias:** some groups are more likely to be included.
- **Non-response bias:** people who do not respond differ from those who do.
- **Survivorship bias:** only cases that remain visible are studied.
- **Convenience sampling:** sampling whoever is easiest to reach.

The **Central Limit Theorem (CLT)** says that, under common conditions, the distribution of sample means approaches a normal distribution as sample size grows, even when the original population is not normal. It does not say the raw data become normally distributed.

In [ ]:
rng = np.random.default_rng(10)
# Skewed population: exponential rather than normal.
skewed_population = rng.exponential(scale=1.0, size=200_000)

means_n5 = [rng.choice(skewed_population, size=5).mean() for _ in range(3000)]
means_n40 = [rng.choice(skewed_population, size=40).mean() for _ in range(3000)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(means_n5, bins=35)
axes[0].set_title("Sample means (n=5)")
axes[1].hist(means_n40, bins=35)
axes[1].set_title("Sample means (n=40)")
plt.tight_layout()
plt.show()

## 7. Standard error and confidence intervals

**Standard deviation** describes variability among individual observations. **Standard error (SE)** describes the variability of a statistic across repeated samples.

For a sample mean, a common estimated standard error is:

\[
SE(\bar{x})=\frac{s}{\sqrt{n}}
\]

A rough 95% confidence interval for a mean with a sufficiently large sample is:

\[
\bar{x}\pm1.96\times SE(\bar{x})
\]

A 95% confidence procedure means that, over repeated samples under its assumptions, about 95% of intervals constructed this way would contain the true parameter. It does not mean there is a 95% frequentist probability that this one fixed interval contains the fixed parameter.

In [ ]:
rng = np.random.default_rng(21)
sample = rng.normal(loc=250, scale=40, size=100)

mean = sample.mean()
se = sample.std(ddof=1) / np.sqrt(len(sample))
lower = mean - 1.96 * se
upper = mean + 1.96 * se

print(f"Sample mean: {mean:.2f}")
print(f"Standard error: {se:.2f}")
print(f"Approximate 95% CI: ({lower:.2f}, {upper:.2f})")

## 8. Hypothesis testing and p-values

A hypothesis test evaluates how compatible the observed data are with a null model.

- **Null hypothesis (H₀):** often says there is no difference or no effect.
- **Alternative hypothesis (H₁):** describes a difference or effect of interest.
- **Significance level (α):** a preselected decision threshold, often 0.05.
- **p-value:** assuming H₀ and the test assumptions are true, the probability of seeing results at least as extreme as the observed result.

A p-value is **not**:
- the probability that H₀ is true;
- the probability the result happened “by chance” in a general sense;
- a measure of the size or business importance of an effect.

Statistical significance and practical importance are different questions.

## 9. A two-sample t-test with SciPy

We compare two independent groups' average outcomes. The Welch t-test does not assume equal population variances. A test result should be interpreted alongside the group means, confidence interval, sample sizes, and assumptions.

If SciPy is not installed in your environment, install it with `pip install scipy` or run this notebook in an environment where it is available.

In [ ]:
from scipy import stats

rng = np.random.default_rng(99)
group_a = rng.normal(loc=100, scale=15, size=80)
group_b = rng.normal(loc=106, scale=18, size=75)

test_result = stats.ttest_ind(group_a, group_b, equal_var=False)
difference = group_b.mean() - group_a.mean()

print(f"Mean A: {group_a.mean():.2f}")
print(f"Mean B: {group_b.mean():.2f}")
print(f"Difference (B - A): {difference:.2f}")
print(f"Welch t-statistic: {test_result.statistic:.3f}")
print(f"Two-sided p-value: {test_result.pvalue:.4f}")

### A good interpretation template

1. State the observed difference in original units.
2. State the p-value and the threshold selected in advance.
3. If possible, report a confidence interval for the difference.
4. Describe practical significance and possible confounding.
5. Avoid saying the test proves the null hypothesis or proves causation.

## 10. Type I error, Type II error, and power

- **Type I error:** reject a true null hypothesis (false positive).
- **Type II error:** fail to reject a false null hypothesis (false negative).
- **Power:** probability of rejecting H₀ when a specified alternative is true; `power = 1 - β`.

Power generally increases with larger sample size, a larger true effect, lower variability, or a higher significance level. Raising α also increases the Type I error rate, so it is a trade-off rather than a free improvement.

## 11. Effect size

A p-value alone does not tell you how large an effect is. Always consider an effect-size measure in context.

For two groups:
- **Raw mean difference:** interpretable in the outcome's units.
- **Relative change:** `(new - old) / old`, when the baseline is meaningful and nonzero.
- **Cohen's d:** mean difference divided by a pooled standard deviation (under its assumptions).

A tiny effect can be statistically significant with a very large sample. A meaningful effect can fail to reach significance when the sample is too small or noisy.

In [ ]:
mean_a, mean_b = group_a.mean(), group_b.mean()
sd_a, sd_b = group_a.std(ddof=1), group_b.std(ddof=1)
n_a, n_b = len(group_a), len(group_b)

pooled_sd = np.sqrt(((n_a - 1) * sd_a**2 + (n_b - 1) * sd_b**2) / (n_a + n_b - 2))
cohens_d = (mean_b - mean_a) / pooled_sd

print(f"Raw mean difference (B - A): {mean_b - mean_a:.2f}")
print(f"Relative change vs A: {(mean_b - mean_a) / mean_a:.2%}")
print(f"Cohen's d: {cohens_d:.3f}")

## 12. Correlation does not imply causation

Correlation describes how two variables move together. It does not, by itself, establish that one causes the other.

Possible explanations for an association:
- A causes B.
- B causes A.
- A third variable influences both (confounding).
- Selection or measurement bias creates an association.
- The pattern is due to random variation.

A causal claim needs an appropriate design and assumptions—for example, a randomized experiment or a carefully justified observational analysis.

In [ ]:
rng = np.random.default_rng(15)
hours_studied = rng.uniform(0, 10, 300)
prior_knowledge = rng.normal(50, 10, 300)
score = 30 + 4 * hours_studied + 0.7 * prior_knowledge + rng.normal(0, 8, 300)

df = pd.DataFrame({
    "hours_studied": hours_studied,
    "prior_knowledge": prior_knowledge,
    "score": score,
})
print("Correlation: hours studied vs score =", round(df["hours_studied"].corr(df["score"]), 3))
print("Correlation: prior knowledge vs score =", round(df["prior_knowledge"].corr(df["score"]), 3))
df.head()

## 13. Covariance and linear regression foundations

- **Covariance** indicates whether two variables tend to move together, but its magnitude depends on their units.
- **Correlation** standardizes covariance and ranges from −1 to +1.
- **Linear regression** models an outcome as a linear combination of predictors, often written as `y = β₀ + β₁x + ε`.
- **Slope (β₁):** expected change in the model's predicted outcome for a one-unit increase in x, holding other included predictors fixed in a multiple regression.
- **Residual:** observed value minus predicted value.

Regression describes associations unless the study design and assumptions support a causal interpretation. Inspect residuals, outliers, nonlinearity, and whether uncertainty estimates are appropriate.

In [ ]:
x = df["hours_studied"].to_numpy()
y = df["score"].to_numpy()

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
predicted = intercept + slope * x
residuals = y - predicted

print(f"Intercept: {intercept:.2f}")
print(f"Slope: {slope:.2f} score points per study hour")
print(f"Correlation r: {r_value:.3f}")
print(f"Regression p-value for slope: {p_value:.4g}")
print(f"Residual mean (approximately 0 with intercept): {residuals.mean():.6f}")

## 14. Outliers and robust analysis

An outlier is an observation that is far from the bulk of the data under some definition. It may be:
- a data-entry or measurement error;
- a valid but rare event;
- evidence that groups or processes differ.

Do not automatically delete outliers. Investigate the source, compare robust summaries such as the median and IQR, and understand whether the business question includes rare high-impact events.

In [ ]:
values = pd.Series([10, 11, 11, 12, 12, 13, 14, 15, 80])
q1, q3 = values.quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
potential_outliers = values[(values < lower_fence) | (values > upper_fence)]

print("Mean:", values.mean())
print("Median:", values.median())
print("IQR:", iqr)
print("Potential outliers by 1.5×IQR rule:", potential_outliers.tolist())

## 15. Practice questions — try before reading the solutions

1. A dataset contains costs `[10, 12, 12, 14, 100]`. Which is larger: mean or median, and why?
2. If `P(A)=0.2`, what is `P(not A)`?
3. A condition has 1% prevalence. A test has 90% sensitivity and 95% specificity. Why is a positive result not a 90% probability of having the condition?
4. Explain the difference between standard deviation and standard error.
5. In plain language, what does a p-value of 0.03 mean if the null hypothesis and test assumptions are true?
6. What is the difference between statistical significance and practical significance?
7. Why does a large correlation not necessarily mean one variable causes the other?
8. Why might the median be a better summary than the mean for highly skewed costs?

### Suggested solutions

1. Mean = 29.6, median = 12. The value 100 pulls the mean upward.
2. `1 - 0.2 = 0.8`.
3. Sensitivity is `P(positive | condition)`, not `P(condition | positive)`. Prevalence and false positives also matter.
4. SD describes spread among observations; SE describes uncertainty/variability of a statistic such as the sample mean.
5. Under the null model and test assumptions, data at least as extreme as these would occur with probability 3%. It is not the probability that the null is true.
6. Statistical significance concerns evidence relative to a model and threshold; practical significance concerns whether the effect is large or important enough to matter.
7. Confounding, reverse direction, bias, or coincidence may explain the association.
8. The median is less sensitive to extreme values and can better represent a typical observation in skewed data.

## 16. Mini-project — evaluate a hypothetical intervention

**Scenario:** A team wants to know whether a new outreach process is associated with lower average monthly cost. You receive two independent groups of synthetic observations.

Tasks:
1. Describe each group's sample size, mean, median, and standard deviation.
2. Calculate the difference in group means.
3. Perform a Welch two-sample t-test.
4. Explain the p-value in non-technical language.
5. Discuss practical importance and limitations.
6. Explain why an observational difference alone does not prove the outreach caused the change.

Do not decide that the intervention works from the p-value alone.

In [ ]:
rng = np.random.default_rng(2026)
usual_care = rng.gamma(shape=4.5, scale=120, size=120)
outreach = rng.gamma(shape=4.2, scale=120, size=125)

project_summary = pd.DataFrame({
    "usual_care": [
        len(usual_care), usual_care.mean(), np.median(usual_care), usual_care.std(ddof=1)
    ],
    "outreach": [
        len(outreach), outreach.mean(), np.median(outreach), outreach.std(ddof=1)
    ],
}, index=["n", "mean", "median", "sample_sd"])
project_summary.round(2)

In [ ]:
project_test = stats.ttest_ind(outreach, usual_care, equal_var=False)
mean_difference = outreach.mean() - usual_care.mean()
print(f"Mean difference (outreach - usual care): {mean_difference:.2f}")
print(f"Welch t-test p-value: {project_test.pvalue:.4f}")
print("Interpret the size, uncertainty, design, and possible confounding before making a conclusion.")

## 17. Readiness checklist

- [ ] Explain population, sample, parameter, and statistic.
- [ ] Calculate and interpret mean, median, variance, SD, and IQR.
- [ ] Explain probability and conditional probability.
- [ ] Apply Bayes' theorem to a rare-event example.
- [ ] Describe common distributions and their assumptions.
- [ ] Explain sampling bias and the Central Limit Theorem.
- [ ] Distinguish standard deviation from standard error.
- [ ] Interpret a confidence interval correctly.
- [ ] Explain a p-value without saying it is the probability the null is true.
- [ ] Distinguish Type I error, Type II error, and power.
- [ ] Explain effect size and practical significance.
- [ ] Explain why correlation does not establish causation.
- [ ] Describe regression slopes and residuals at a basic level.

### Next module
**Module 5 — Machine Learning Foundations:** problem framing, train/validation/test splits, leakage, preprocessing pipelines, baseline models, cross-validation, classification and regression metrics, and model evaluation.